# Graphify knowledge graphs — 2026 SAGE summer camp Hermes brains

**Purpose:** reproducible notes of how we harvested student Hermes `sage` brains, cleaned them for public share, and built **curated (viz)** vs **full (query)** Graphify knowledge graphs for the camp baseline (`hermes-profile`) and each public student archive.

**Audience:** paper / poster / blog write-up; instructors re-running the pipeline.

**Repos**

| Role | Path |
| --- | --- |
| Student archives + scripts | `2026-summer-camp-sage-agent-logs/hermes/` |
| Camp distribution repo | [`waggle-sensor/summer-camp-2026`](https://github.com/waggle-sensor/summer-camp-2026) |
| Camp distribution profile | [`hermes-profile/`](https://github.com/waggle-sensor/summer-camp-2026/tree/main/hermes-profile) in that repo |
| Public student tarballs | `foundry/harvest/1_4_0/brains/<node>/*_public_sage.tar.gz` |
| Baseline update script | [`scripts/update_hermes_profile_graphify.sh`](https://github.com/waggle-sensor/summer-camp-2026/blob/main/scripts/update_hermes_profile_graphify.sh) |
| Public batch update script | `foundry/tools/update_public_graphify.sh` |

**Two-graph design (important)**

| Scope | Ignore file | Intended use | On-disk name after packaging |
| --- | --- | --- | --- |
| `curated` | `.graphifyignore.curated` | Readable HTML / report (`agent-knowledge-graph.html`) | `graphify-out-viz/` |
| `full` | `.graphifyignore.full` | Agent query / deep GraphRAG | `graphify-out/` |

Do **not** ship the full-graph HTML as the “main” student-facing viz — keep curated `graph.html` as the visible overview.

## Timeline (from lab notes)

### Fri 2026-07-31 — harvest kickoff
- Summer camp / Sage all-hands week; audited student Hermes brain uploads.
- Many students had not pushed brains to GitHub → harvested from nodes where still reachable.
- Some nodes already disconnected (lost those).
- Collection report (Codex project): Hermes brains 2026-07-31.
- **Lesson for write-up:** record harvest steps carefully for paper/poster/blog.

### Mon–Wed 2026-08-03 … 08-05 — cleanup & packaging
- Clean private → public archives: drop bulk/runtime noise, redact secrets.
- Scripts: `clean_public_archive.sh`, `redact_secrets.py`.
- Goal: trees useful for Graphify + merging into camp `hermes-profile`.

### Thu 2026-08-06 — finding the right Graphify command
- Early automation was too slow / wrong path:
  - Full re-extract when we meant incremental update.
  - Agent `/graphify --update` playbook → mostly AST-only, not semantic LLM extraction.
- **Correct CLI:** `graphify extract` — uses existing graph when present (incremental), runs **AST + semantic** extraction.
- NRP ellm: frequent connection / rate-limit style failures → moved inference to **NVIDIA Brev**.

### Fri 2026-08-07 — Brev NIM + curated vs full scopes
- Deployed H200 + NIM; started with `meta/llama-3.3-70b-instruct`, then switched to **`google/gemma-4-31B-it`** (fits H200 better → higher concurrency / token budget; fewer context-split BadRequestErrors).
- Regenerated from scratch (NRP-era graphs were untrustworthy).
- Full baseline HTML was unreadable (~10k+ AST-heavy nodes / hundreds of communities).
- Introduced **curated** `.graphifyignore` (skill cards / `SKILL.md` / camp docs; drop `scripts/`, `references/`, validators, etc.) vs **full** ignore for query graphs.
- Instructor scripts gained `--scope curated|full`, `--cluster-only`, packing helpers.

### Mon–Tue 2026-08-10 … 08-11 — dual graphs shipped
- Built curated baseline → renamed to `graphify-out-viz` → copied into student brains → batch curated student graphs → rename viz folders.
- Built full baseline → copied into student `graphify-out/` → batch full student graphs → repack tarballs.
- Public layout: share `*_public_sage.tar.gz`; extracted `sage/` dirs removed from git working tree after pack (gitignored anyway).

## Pipeline overview

```text
Nodes / GitHub
    │ harvest Hermes sage profiles
    ▼
foundry/.work/private/  ──clean + redact──►  foundry/harvest/1_4_0/brains/*_public_sage.tar.gz
                                              │
                     ┌────────────────────────┴────────────────────────┐
                     ▼                                                 ▼
         Wisp (repo root)                      per-student sage/
                     │                                                 │
         update_hermes_profile_graphify.sh                   update_public_graphify.sh
                     │                                                 │
         ┌───────────┴───────────┐                       ┌────────────┴────────────┐
         ▼                       ▼                       ▼                         ▼
   --scope curated         --scope full          --scope curated            --scope full
   graphify-out-viz/       graphify-out/         graphify-out-viz/          graphify-out/
   (HTML / report)         (agent query)         (HTML / report)            (agent query)
```

**Typical extract knobs (Brev + Gemma on H200):**

- `--max-concurrency 12`
- `--token-budget 50000`
- `--viz-node-limit 20000`
- `--resolution 0.3` (Leiden; lower → fewer/larger communities — still secondary to corpus size)
- Model: `google/gemma-4-31B-it` via `OPENAI_BASE_URL=http://localhost:8000/v1`

## 0. Environment (Brev NIM)

Two steps, and the first one was missing from earlier revisions of this notebook:
**deploy** the NIM on the Brev instance, then **tunnel** to it. Recording only the
tunnel made the environment unreproducible — a later rebuild stalled until the
docker login/pull/run recipe was reconstructed from scratch.

Keep the port-forward up for the whole extract. Helpers live under `foundry/tools/`.

In [ ]:
# Shell cells: run from a terminal (or %%bash). Paths assume sibling checkouts:
#   ~/Git/2026-summer-camp-sage-agent-logs
#   ~/Git/summer-camp-2026

from pathlib import Path

LOGS = Path.home() / "Git/2026-summer-camp-sage-agent-logs"
CAMP = Path.home() / "Git/summer-camp-2026"
HERMES = LOGS / "hermes"
PROFILE = CAMP / "hermes-profile"
ARCHIVES = HERMES / "public/archives"

assert HERMES.is_dir(), HERMES
assert PROFILE.is_dir(), PROFILE
print("logs:", LOGS)
print("camp:", CAMP)
print("archives:", ARCHIVES)
print("students:", sorted(p.name for p in ARCHIVES.iterdir() if p.is_dir()))

### 0a. Deploy the NIM (once per instance boot)

```bash
# One command; waits for /v1/health/ready and prints the served model id.
./foundry/tools/start_nim_on_brev.sh --instance nim-h200 --key-file ~/.ngc/apikey
```

What it does, if you need to run it by hand:

```bash
brev exec nim-h200 "
  echo \"\$NGC_API_KEY\" | docker login nvcr.io -u '\$oauthtoken' --password-stdin
  docker run -d --name nim --gpus all --shm-size=16g -e NGC_API_KEY \
    -v \"\$HOME/.cache/nim:/opt/nim/.cache\" -u \"\$(id -u)\" -p 8000:8000 \
    nvcr.io/nim/google/gemma-4-31b-it:1.7.1-variant
"
```

**Gotchas that cost real time:**

- **Pin the tag.** `1.7.1-variant` is the tag that works for gemma-4-31b-it.
  List what is actually available via the catalog API, not the docker registry —
  the registry token endpoint returns a bare `401`:
  ```bash
  curl -s -H "Authorization: ApiKey $NGC_API_KEY" \
    https://api.ngc.nvidia.com/v2/repos/nim/google/gemma-4-31b-it/images
  ```
- **The model id is case-sensitive.** The image name is lowercase
  (`gemma-4-31b-it`) but the *served* id is **`google/gemma-4-31B-it`** with a
  capital B. Requesting the lowercase form returns
  `404 The model 'google/gemma-4-31b-it' does not exist.`
- **Cold start is ~10–15 min** while weights download and load. Mounting
  `~/.cache/nim` makes the next boot much faster.
- **Never pass the NGC key as a CLI argument** — it lands in shell history and
  in `ps` on the instance. Use a `600` key file or the environment.
- **Stop the instance when the extract finishes.** An H200 bills ~$5.40/hr:
  `./foundry/tools/stop_nim_after_graphify.sh --instance nim-h200`

### 0b. Tunnel to it

```bash
# Terminal A — NIM on Brev → localhost:8000
brev port-forward nim-h200 -p 8000:8000
# Better for long extracts — restarts the forward only when the endpoint stops
# responding:
./foundry/tools/watch_nim_port_forward.sh --instance nim-h200 --port 8000

export OPENAI_BASE_URL=http://localhost:8000/v1
export OPENAI_API_KEY=not-needed
export OPENAI_MODEL=google/gemma-4-31B-it
curl -s "$OPENAI_BASE_URL/models" | head
```

**Do not write your own reconnect loop.** `brev port-forward` exits once the SSH
LocalForward is established while the `ssh` child keeps listening, so a naive
`while true; do ssh -L ...; done` cannot bind the port that the *healthy* tunnel
already holds. It then respawns every couple of seconds forever, and the
connection churn tears down in-flight completions mid-request — an extract goes
silent while the GPU still reads 100% busy. `watch_nim_port_forward.sh` polls
health and only intervenes when the endpoint is genuinely down.

## 1. Harvest → private → public (context)

High level only (details in private audit docs):

1. Collect Hermes `sage` profiles from student nodes / repos into `foundry/.work/private/`.
2. Rebuild public shareables:

```bash
cd ~/Git/2026-summer-camp-sage-agent-logs
./foundry/tools/clean_public_archive.sh --all
```

3. Unpack when you need a live `sage/` tree for Graphify:

```bash
for d in foundry/harvest/1_4_0/brains/*/; do
  s=$(basename "$d")
  tar -xzf "$d/${s}_public_sage.tar.gz" -C "$d"
done
```

## 2. Camp baseline — CURATED (viz)

Installs `.graphifyignore.curated` → `.graphifyignore`, runs extract + cluster, packs `graphify-baseline.tar.gz` + `agent-knowledge-graph.html`.

```bash
cd ~/Git/summer-camp-2026

./scripts/update_hermes_profile_graphify.sh \
  --wipe \
  --pack-baseline \
  --scope curated \
  --max-concurrency 12 \
  --token-budget 50000 \
  --viz-node-limit 20000 \
  --resolution 0.3

# Rename for packaging: curated graph is viz-only (not the agent query root)
mv hermes-profile/graphify-out hermes-profile/graphify-out-viz
# Optional: keep agent-knowledge-graph.html pointing at curated HTML
cp hermes-profile/graphify-out-viz/graph.html hermes-profile/agent-knowledge-graph.html
```

## 3. Student archives — CURATED (viz)

Seed each student from the curated baseline (optional but what we did), then batch extract with `--scope curated`.

```bash
LOGS=~/Git/2026-summer-camp-sage-agent-logs
SRC=~/Git/Wisp (repo root)/graphify-out-viz   # or graphify-out if not renamed yet
ARCH=$LOGS/foundry/harvest/1_4_0/brains

# Copy curated baseline into each sage/ as graphify-out (extract target name),
# then run the batch script (or rename to graphify-out-viz after extract).
for sage in "$ARCH"/*/sage; do
  rm -rf "$sage/graphify-out"
  cp -a "$SRC" "$sage/graphify-out"
done

cd "$LOGS"
./foundry/tools/update_public_graphify.sh --all \
  --scope curated \
  --max-concurrency 12 \
  --token-budget 50000 \
  --viz-node-limit 20000 \
  --resolution 0.3 \
  --repack

# After SUCCESS: rename student viz graphs
for sage in "$ARCH"/*/sage; do
  [[ -d "$sage/graphify-out" ]] || continue
  rm -rf "$sage/graphify-out-viz"
  mv "$sage/graphify-out" "$sage/graphify-out-viz"
done

# Re-pack if rename happened after --repack
for d in "$ARCH"/*/; do
  s=$(basename "$d")
  tar -czf "$d/${s}_public_sage.tar.gz" -C "$d" sage
done
```

Reran individually after connection errors: `node-H01E`, `node-H03F`.

## 4. Camp baseline — FULL (query)

Whole-codebase ignore (still drops evals / media / vendor noise).

```bash
cd ~/Git/summer-camp-2026

./scripts/update_hermes_profile_graphify.sh \
  --wipe \
  --pack-baseline \
  --scope full \
  --max-concurrency 12 \
  --token-budget 50000 \
  --viz-node-limit 20000 \
  --resolution 0.3

# Keep curated HTML as the published “main” viz if pack-baseline overwrote it:
cp hermes-profile/graphify-out-viz/graph.html hermes-profile/agent-knowledge-graph.html
```

## 5. Student archives — FULL (query)

Copy full baseline `graphify-out/` into each student, then batch with `--scope full`. Leave `graphify-out-viz/` untouched.

```bash
LOGS=~/Git/2026-summer-camp-sage-agent-logs
SRC=~/Git/Wisp (repo root)/graphify-out
ARCH=$LOGS/foundry/harvest/1_4_0/brains

for sage in "$ARCH"/*/sage; do
  rm -rf "$sage/graphify-out"
  cp -a "$SRC" "$sage/graphify-out"
done

cd "$LOGS"
./foundry/tools/update_public_graphify.sh --all \
  --scope full \
  --max-concurrency 12 \
  --token-budget 50000 \
  --viz-node-limit 20000 \
  --resolution 0.3 \
  --repack

# Ensure published HTML in each tree (if present) tracks curated viz, not full graph.html
for sage in "$ARCH"/*/sage; do
  if [[ -f "$sage/graphify-out-viz/graph.html" ]]; then
    cp "$sage/graphify-out-viz/graph.html" "$sage/agent-knowledge-graph.html"
  fi
done

for d in "$ARCH"/*/; do
  s=$(basename "$d")
  [[ -d "$d/sage" ]] || continue
  tar -czf "$d/${s}_public_sage.tar.gz" -C "$d" sage
done

# Optional: drop extracted sage/ after pack (tarballs are the shareable artifact)
for d in "$ARCH"/*/; do
  rm -rf "$d/sage"
done
```

## 6. Cluster-only / repair helpers

When `graph.json` already exists and you only need Leiden + `GRAPH_REPORT.md` / HTML refresh:

```bash
./foundry/tools/update_public_graphify.sh --all \
  --cluster-only \
  --resolution 0.3 \
  --viz-node-limit 20000 \
  --repack

# Single student retry after a failed extract
./foundry/tools/update_public_graphify.sh node-H01E \
  --scope curated \
  --max-concurrency 12 \
  --token-budget 50000 \
  --viz-node-limit 20000 \
  --resolution 0.3 \
  --repack
```

## 7. Inspect a packed student tarball

In [ ]:
import tarfile
from collections import Counter

student = "node-H01D"
tar_path = ARCHIVES / student / f"{student}_public_sage.tar.gz"
assert tar_path.is_file(), tar_path

names = []
with tarfile.open(tar_path, "r:gz") as tf:
    names = [m.name for m in tf.getmembers() if m.isdir() or m.isfile()]

tops = Counter()
for n in names:
    parts = n.split("/")
    if len(parts) >= 2 and parts[0] == "sage":
        tops[parts[1]] += 1

print(tar_path)
print("members:", len(names))
print("top-level under sage/:")
for k, v in tops.most_common(20):
    print(f"  {k:30s} {v}")
print("has graphify-out:", any("sage/graphify-out/" in n or n == "sage/graphify-out" for n in names))
print("has graphify-out-viz:", any("sage/graphify-out-viz/" in n or n == "sage/graphify-out-viz" for n in names))

## Lessons learned (for paper / poster / blog)

1. **Harvest early** — nodes go offline; GitHub alone under-reported contributions.
2. **`graphify extract` is the real update path** — not “from scratch only”; it also drives semantic LLM extraction. AST-only `/graphify --update` is insufficient for doc-heavy Hermes skills.
3. **Inference hosting matters** — NRP ellm connection errors corrupted trust in early graphs; Brev H200 + NIM was stable enough to wipe and rebuild.
4. **Model size vs throughput** — Gemma 4 31B on H200 allowed higher concurrency/token budget than Llama 3.3 70B and reduced context-split retries.
5. **Corpus > Leiden resolution** — full skill trees (scripts/validators) produce huge AST graphs; lowering `--resolution` alone cannot make HTML readable. Need a **curated ignore** for viz and a **full** graph for querying.
6. **Name the artifacts** — `graphify-out-viz` (human) vs `graphify-out` (agent) prevents shipping the wrong HTML as “main”.
7. **Tarballs are the shareable unit** — extracted `sage/` is a local workspace; pack after edits, then delete extracts if desired.

### Artifact checklist

- [ ] Private harvest audit + public clean/redact audits
- [ ] `.graphifyignore.curated` / `.graphifyignore.full`
- [ ] Camp `agent-knowledge-graph.html` = curated viz
- [ ] Camp + student `graphify-out-viz/` (curated)
- [ ] Camp + student `graphify-out/` (full)
- [ ] Public `*_public_sage.tar.gz` updated and pushed
- [ ] Logs: `foundry/.work/graphify-update-logs/`, `scripts/graphify-update-logs/hermes-profile.log`

9. **Document the deploy, not just the connection.** This notebook originally
   recorded the port-forward but not how the NIM behind it got there. The next
   rebuild stalled on exactly that gap. Captured now as
   `foundry/tools/start_nim_on_brev.sh`.
10. **Incremental beats `--wipe`.** `--wipe` deletes `graphify-out/` *including
    its semantic cache*, which is why the baseline cost ~$1.02 for 1,421 files.
    If the prompt fingerprint (`cache/semantic/p<fp>/`) is unchanged, extract
    incrementally instead. Note the manifest only records files that
    *contributed nodes*, so files that produced none re-extract every time —
    budget for that rather than being surprised by it.
11. **`graphify extract` always writes to `<profile>/graphify-out/`.** There is
    no output-dir flag, so maintaining two graphs means swapping directories.
    Stash the idle graph **outside** the profile — a sibling like
    `graphify-out-full/` is matched by no `.graphifyignore` rule, and graphify
    will happily scan the stashed graph's own cache as source material.

## Related docs

- [`waggle-sensor/summer-camp-2026`](https://github.com/waggle-sensor/summer-camp-2026) — camp workspace + `hermes-profile/`
- `hermes/README.md` — unpack / repack / Graphify CLI
- `foundry/.work/graphify-update-audit-summary.md` — last batch audit
- [`.graphifyignore.curated`](https://github.com/waggle-sensor/summer-camp-2026/blob/main/hermes-profile/.graphifyignore.curated)
- [`.graphifyignore.full`](https://github.com/waggle-sensor/summer-camp-2026/blob/main/hermes-profile/.graphifyignore.full)